# Optuna, Evaluation, and Export

Notebook ini melakukan tuning dan evaluasi ranking di atas candidate pool yang lebih sulit, tetap memakai dataset raw yang sudah diprepare.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import joblib
import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMRanker, early_stopping, log_evaluation
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from src.config import ARTIFACTS_DIR, MODELS_DIR, INTERIM_DIR
from src.evaluation import evaluate_grouped_ndcg, grouped_ranking_metrics, precision_at_k
from src.features import build_candidate_features, compute_relevance, feature_columns, synthetic_user_profiles
from src.preprocessing import load_prepared_dataset, save_dataframe


c:\porto\NemuParfang\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_prepared_dataset(INTERIM_DIR)
embeddings = joblib.load(ARTIFACTS_DIR / 'perfume_embeddings.joblib')
with (ARTIFACTS_DIR / 'embedding_config.json').open('r', encoding='utf-8') as handle:
    embedding_config = json.load(handle)
encoder = SentenceTransformer(embedding_config['model_name'])
rng = np.random.default_rng(42)

print('Loaded perfumes:', len(df))
print('Embedding shape:', embeddings.shape)
print('Embedding model:', embedding_config['model_name'])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8812.28it/s]


Loaded perfumes: 70103
Embedding shape: (70103, 384)
Embedding model: all-MiniLM-L6-v2


In [3]:
def expand_profiles(base_profiles):
    variants = [
        ('core', ''),
        ('refined', ' slightly more refined'),
        ('everyday', ' everyday wear'),
        ('bold', ' stronger more expressive'),
        ('minimal', ' cleaner more minimal style'),
    ]
    expanded = []
    for profile in base_profiles:
        family = profile.get('profile_family', profile['profile_id'])
        for suffix, extra_text in variants:
            expanded.append({
                **profile,
                'profile_id': f"{family}_{suffix}",
                'profile_family': family,
                'profile_text': profile['profile_text'] + extra_text,
            })
    return expanded


def build_rank_frame(df, embeddings, encoder, profiles, top_k=100, hard_k=120, random_k=100, noise_rate=0.2, seed=42):
    rng_local = np.random.default_rng(seed)
    frames = []
    all_indices = np.arange(len(df))

    for profile in profiles:
        query_vec = encoder.encode(profile['profile_text'])
        scores = cosine_similarity([query_vec], embeddings).ravel()
        ranked_indices = np.argsort(scores)[::-1]
        top_indices = ranked_indices[:top_k]
        hard_pool = ranked_indices[top_k:top_k + 400]
        hard_take = min(hard_k, len(hard_pool))
        hard_indices = rng_local.choice(hard_pool, size=hard_take, replace=False) if hard_take else np.array([], dtype=int)
        remaining_indices = np.setdiff1d(all_indices, np.concatenate([top_indices, hard_indices]), assume_unique=False)
        random_take = min(random_k, len(remaining_indices))
        random_indices = rng_local.choice(remaining_indices, size=random_take, replace=False) if random_take else np.array([], dtype=int)
        candidate_indices = np.unique(np.concatenate([top_indices, hard_indices, random_indices]))

        candidates = df.iloc[candidate_indices].copy().reset_index(drop=True)
        candidates['profile_id'] = profile['profile_id']
        candidates['profile_family'] = profile['profile_family']
        candidates['cosine_similarity_score'] = scores[candidate_indices]
        candidates = build_candidate_features(candidates, profile)
        candidates['relevance'] = candidates.apply(lambda row: compute_relevance(profile, row), axis=1)

        if noise_rate > 0:
            noise = rng_local.choice([-1, 0, 1], size=len(candidates), p=[noise_rate / 2, 1 - noise_rate, noise_rate / 2])
            candidates['relevance'] = np.clip(candidates['relevance'] + noise, 0, 4)

        frames.append(candidates)

    return pd.concat(frames, ignore_index=True)


profiles = expand_profiles(synthetic_user_profiles())
rank_df = build_rank_frame(df, embeddings, encoder, profiles, top_k=100, hard_k=120, random_k=100, noise_rate=0.25)
display(rank_df[['profile_id', 'profile_family', 'perfume_name', 'brand', 'country', 'cosine_similarity_score', 'relevance']].head(15))
print('Relevance distribution:')
display(rank_df['relevance'].value_counts().sort_index())
print('Profiles:', rank_df['profile_id'].nunique())
print('Profile families:', rank_df['profile_family'].nunique())
print('Rows:', len(rank_df))


,profile_id,profile_family,perfume_name,brand,country,cosine_similarity_score,relevance
0,woody_evening_core,woody_evening,The Pride Of Armaf Admiral,Armaf,United Arab Emirates,0.547750,3
1,woody_evening_core,woody_evening,Aures,Avon,United States,0.694916,2
2,woody_evening_core,woody_evening,Cordovan,Avon,United States,0.650589,1
3,woody_evening_core,woody_evening,Sassy Swirls Vanilla Bean,Avon,United States,0.688209,2
4,woody_evening_core,woody_evening,Eve Duet Radiant,Avon,United States,0.410790,1
5,woody_evening_core,woody_evening,Full Speed Nitro,Avon,United States,0.695302,1
6,woody_evening_core,woody_evening,Cherry Mousse,Avon,United States,0.451013,2
7,woody_evening_core,woody_evening,Azzaro Pour Homme Limited Edition 2014,Azzaro,NaN,0.692069,1
8,woody_evening_core,woody_evening,Single Barrel Bourbon,Bath & Body Works,United States,0.675672,2
9,woody_evening_core,woody_evening,Strawberry Snowflakes,Bath & Body Works,United States,0.482130,1


Relevance distribution:


relevance
0    4884
1    4633
2    7723
3    5256
4    1504
Name: count, dtype: int64

Profiles: 75
Profile families: 15
Rows: 24000


In [4]:
saved_path = save_dataframe(rank_df, ARTIFACTS_DIR / 'ranker_training_frame.csv')
print('Saved training frame to:', saved_path)


Saved training frame to: C:\porto\NemuParfang\ml_training_notebooks\artifacts\ranker_training_frame.csv


In [5]:
profile_families = rank_df['profile_family'].unique().tolist()
train_families, val_families = train_test_split(profile_families, test_size=0.33, random_state=42)
train_df = rank_df[rank_df['profile_family'].isin(train_families)].copy()
val_df = rank_df[rank_df['profile_family'].isin(val_families)].copy()
train_groups = train_df.groupby('profile_id').size().tolist()
val_groups = val_df.groupby('profile_id').size().tolist()
print('Train rows:', len(train_df), 'Validation rows:', len(val_df))
print('Train groups:', len(train_groups), 'Validation groups:', len(val_groups))

def objective(trial):
    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'num_leaves': trial.suggest_int('num_leaves', 16, 63),
        'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.08, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 900),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 80),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 2.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 2.0, log=True),
        'force_col_wise': True,
        'random_state': 42,
    }

    model = LGBMRanker(**params)
    model.fit(
        train_df[feature_columns()],
        train_df['relevance'],
        group=train_groups,
        eval_set=[(val_df[feature_columns()], val_df['relevance'])],
        eval_group=[val_groups],
        eval_at=[5, 10],
        callbacks=[early_stopping(40), log_evaluation(0)],
    )
    return evaluate_grouped_ndcg(model, val_df[feature_columns()], val_df['relevance'].to_numpy(), val_groups, k=10)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, timeout=1800)
study.best_params, study.best_value


[I 2026-05-31 23:03:58,447] A new study created in memory with name: no-name-610461c5-4c4c-4028-a788-149a8dcd71c0


Train rows: 16000 Validation rows: 8000
Train groups: 50 Validation groups: 25
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 40 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

[I 2026-05-31 23:04:00,584] Trial 0 finished with value: 0.9351635060197836 and parameters: {'num_leaves': 42, 'learning_rate': 0.009619246561274563, 'n_estimators': 516, 'min_child_samples': 25, 'subsample': 0.9815662073973661, 'colsample_bytree': 0.8239987804615792, 'reg_alpha': 1.587348429275984, 'reg_lambda': 0.021441355617255294}. Best is trial 0 with value: 0.9351635060197836.
[I 2026-05-31 23:04:00,716] Trial 1 finished with value: 0.9347696015655473 and parameters: {'num_leaves': 23, 'learning_rate': 0.013076973765385963, 'n_estimators': 407, 'min_child_samples': 73, 'subsample': 0.8311215055303423, 'colsample_bytree': 0.6319384223621567, 'reg_alpha': 0.134365504566929, 'reg_lambda': 0.0014605525828328113}. Best is trial 0 with value: 0.9351635060197836.


[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[6]	valid_0's ndcg@5: 0.91157	valid_0's ndcg@10: 0.879782
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
Training until validation scores don't improve for 40 rounds


[I 2026-05-31 23:04:00,862] Trial 2 finished with value: 0.9403572450202112 and parameters: {'num_leaves': 21, 'learning_rate': 0.022979398962027074, 'n_estimators': 610, 'min_child_samples': 47, 'subsample': 0.6472030180894741, 'colsample_bytree': 0.7094116325990126, 'reg_alpha': 0.000956193004416293, 'reg_lambda': 0.0005667784572027462}. Best is trial 2 with value: 0.9403572450202112.


...

[I 2026-05-31 23:04:03,826] Trial 12 finished with value: 0.9434727898340871 and parameters: {'num_leaves': 34, 'learning_rate': 0.03671195493895069, 'n_estimators': 730, 'min_child_samples': 31, 'subsample': 0.9227086936965271, 'colsample_bytree': 0.7711838262999186, 'reg_alpha': 0.0034618938818184323, 'reg_lambda': 0.00010731730664697889}. Best is trial 8 with value: 0.9473837067678283.


Early stopping, best iteration is:
[18]	valid_0's ndcg@5: 0.92763	valid_0's ndcg@10: 0.886989
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 40 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

[I 2026-05-31 23:04:04,013] Trial 13 finished with value: 0.9217389050365036 and parameters: {'num_leaves': 55, 'learning_rate': 0.016435095410119598, 'n_estimators': 740, 'min_child_samples': 78, 'subsample': 0.9330946391532695, 'colsample_bytree': 0.8681166689189851, 'reg_alpha': 0.04495621171738012, 'reg_lambda': 0.8343976302619395}. Best is trial 8 with value: 0.9473837067678283.
[I 2026-05-31 23:04:04,213] Trial 14 finished with value: 0.948609332993671 and parameters: {'num_leaves': 16, 'learning_rate': 0.07517784044229138, 'n_estimators': 877, 'min_child_samples': 56, 'subsample': 0.7314095983089542, 'colsample_bytree': 0.7056700906739267, 'reg_alpha': 0.00012614556353268095, 'reg_lambda': 0.16417725172889402}. Best is trial 14 with value: 0.948609332993671.


[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[23]	valid_0's ndcg@5: 0.923144	valid_0's ndcg@10: 0.891118


({'num_leaves': 16,
  'learning_rate': 0.07517784044229138,
  'n_estimators': 877,
  'min_child_samples': 56,
  'subsample': 0.7314095983089542,
  'colsample_bytree': 0.7056700906739267,
  'reg_alpha': 0.00012614556353268095,
  'reg_lambda': 0.16417725172889402},
 0.948609332993671)

In [6]:
best_ranker = LGBMRanker(objective='lambdarank', metric='ndcg', random_state=42, **study.best_params)
best_ranker.fit(
    train_df[feature_columns()],
    train_df['relevance'],
    group=train_groups,
    eval_set=[(val_df[feature_columns()], val_df['relevance'])],
    eval_group=[val_groups],
    eval_at=[5, 10],
    callbacks=[early_stopping(40), log_evaluation(50)],
)
val_scores = best_ranker.predict(val_df[feature_columns()])

ndcg10 = evaluate_grouped_ndcg(best_ranker, val_df[feature_columns()], val_df['relevance'].to_numpy(), val_groups, k=10)
precision5 = precision_at_k(val_df['relevance'].to_numpy(), val_scores, k=5)
print('NDCG@10:', ndcg10)
print('Precision@5:', precision5)
group_metrics = grouped_ranking_metrics(val_df['relevance'].to_numpy(), val_scores, val_groups, k=10, threshold=2)
metrics_df = pd.DataFrame([group_metrics])
display(metrics_df)
importance_df = pd.DataFrame({
    'feature': feature_columns(),
    'importance': best_ranker.feature_importances_,
}).sort_values('importance', ascending=False)
display(importance_df)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
Training until validation scores don't improve for 40 rounds
[50]	valid_0's ndcg@5: 0.914973	valid_0's ndcg@10: 0.889795
Early stopping, best iteration is:
[23]	valid_0's ndcg@5: 0.923144	valid_0's ndcg@10: 0.891118
NDCG@10: 0.948609332993671
Precision@5: 1.0


,ndcg@10,precision@10,recall@10,hit_rate@10,mrr@10
0,0.948609,1.0,0.061869,1.0,1.0


,feature,importance
1,accord_overlap_ratio,51
0,cosine_similarity_score,47
8,rating_norm,39
3,note_overlap_ratio,35
4,gender_match,32
12,freshness_score,31
7,year_window_match,28
6,country_match,20
2,accord_overlap_count,17
11,popularity_score,15


In [7]:
def train_and_score(feature_subset, label, params):
    model = LGBMRanker(objective='lambdarank', metric='ndcg', random_state=42, **params)
    model.fit(
        train_df[feature_subset],
        train_df['relevance'],
        group=train_groups,
        eval_set=[(val_df[feature_subset], val_df['relevance'])],
        eval_group=[val_groups],
        eval_at=[5, 10],
        callbacks=[early_stopping(40), log_evaluation(0)],
    )
    preds = model.predict(val_df[feature_subset])
    metrics = grouped_ranking_metrics(val_df['relevance'].to_numpy(), preds, val_groups, k=10, threshold=2)
    metrics['model'] = label
    return model, metrics


all_features = feature_columns()
ablation_sets = {
    'full_model': all_features,
    'no_popularity_freshness': [feature for feature in all_features if feature not in {'popularity_score', 'freshness_score'}],
    'no_metadata_bias': [feature for feature in all_features if feature not in {'rating_norm', 'review_count_norm', 'year_norm', 'popularity_score', 'freshness_score'}],
}

ablation_results = []
for label, features_subset in ablation_sets.items():
    _, metrics = train_and_score(features_subset, label, study.best_params)
    ablation_results.append(metrics)

ablation_df = pd.DataFrame(ablation_results).set_index('model').sort_values('ndcg@10', ascending=False)
display(ablation_df)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001204 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 13
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[23]	valid_0's ndcg@5: 0.923144	valid_0's ndcg@10: 0.891118
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000488 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 807
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 11
Training until validation scores don't improve for 40 rounds
Early stopping, best iteration is:
[4]	valid_0's ndcg@5: 0.922327	valid_0's ndcg@10: 0.890731
[LightGBM] [Info] Auto-choosing row-wise multi-t

,ndcg@10,precision@10,recall@10,hit_rate@10,mrr@10
model,,,,,
full_model,0.948609,1.000,0.061869,1.0,1.0
no_popularity_freshness,0.942666,1.000,0.061869,1.0,1.0
no_metadata_bias,0.939626,0.992,0.061272,1.0,1.0


In [8]:
summary_df = pd.DataFrame([
    {
        'dataset_rows': len(rank_df),
        'train_rows': len(train_df),
        'validation_rows': len(val_df),
        'profile_families': rank_df['profile_family'].nunique(),
        'profile_variants': rank_df['profile_id'].nunique(),
        'best_optuna_ndcg@10': study.best_value,
        'final_model_ndcg@10': metrics_df.loc[0, 'ndcg@10'],
        'final_model_precision@10': metrics_df.loc[0, 'precision@10'],
        'final_model_recall@10': metrics_df.loc[0, 'recall@10'],
        'final_model_hit_rate@10': metrics_df.loc[0, 'hit_rate@10'],
        'final_model_mrr@10': metrics_df.loc[0, 'mrr@10'],
    }
])
display(summary_df)

print('Interpretation guide:')
print('- If full_model is only slightly better than no_popularity_freshness, personalization is healthier.')
print('- If no_metadata_bias collapses badly, the model is still too dependent on popularity and freshness signals.')
print('- NDCG@10 below the original 1.0 with strong hit-rate is a healthier sign than a perfect synthetic score.')


,dataset_rows,train_rows,validation_rows,profile_families,profile_variants,best_optuna_ndcg@10,final_model_ndcg@10,final_model_precision@10,final_model_recall@10,final_model_hit_rate@10,final_model_mrr@10
0,24000,16000,8000,15,75,0.948609,0.948609,1.0,0.061869,1.0,1.0


Interpretation guide:
- If full_model is only slightly better than no_popularity_freshness, personalization is healthier.
- If no_metadata_bias collapses badly, the model is still too dependent on popularity and freshness signals.
- NDCG@10 below the original 1.0 with strong hit-rate is a healthier sign than a perfect synthetic score.


In [9]:
joblib.dump(best_ranker, MODELS_DIR / 'lgbm_ranker_best.joblib')
joblib.dump(study, MODELS_DIR / 'optuna_study.joblib')
print('Final artifacts exported to outputs/models/.')


Final artifacts exported to outputs/models/.
